In [37]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains

import time
import pandas as pd

In [38]:
# page need to be scrapped

url = 'https://go4explore.com/'  

In [39]:
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)


driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()) , options=chrome_options)

driver.get(url)
driver.maximize_window()


In [40]:
# Creating lists 

link = ""
title_link= []
trip_title= []

trip_names = []
p_prices = []
prices= []
durations = []
discounts =[]

In [41]:
# Use explicit wait to wait for the <strong> elements to appear
wait = WebDriverWait(driver, 10)  # Wait for up to 10 seconds

In [42]:
everything = driver.find_elements(By.XPATH, '//div[contains(@class,"mt-4 mt-lg-0 col-4 col-lg-2")]')
for name in everything:
    name_title = name.find_element(By.XPATH, './/p[contains(@class,"mb-0 font_poppins fs_xsm")]').get_attribute("textContent")
    trip_title.append(name_title)
    print(trip_title)
 
print(f"Found {len(everything)} elements")

['Backpacking  Trips']
['Backpacking  Trips', 'Weekend  Getaways']
['Backpacking  Trips', 'Weekend  Getaways', 'International  Trips']
['Backpacking  Trips', 'Weekend  Getaways', 'International  Trips', 'Adventure  Treks']
['Backpacking  Trips', 'Weekend  Getaways', 'International  Trips', 'Adventure  Treks', 'Honeymoon  Trips']
['Backpacking  Trips', 'Weekend  Getaways', 'International  Trips', 'Adventure  Treks', 'Honeymoon  Trips', 'Corporate  Trips']
Found 6 elements


In [43]:
for links in everything:
    link = links.find_element(By.XPATH, './/a[@class=" text-decoration-none d-flex align-items-center justify-content-center flex-column"]').get_attribute("href")
    # print(link)
    title_link.append(link)
    print(title_link)

['https://go4explore.com/trips-category/backpacking-trips']
['https://go4explore.com/trips-category/backpacking-trips', 'https://go4explore.com/trips-category/weekend-trips']
['https://go4explore.com/trips-category/backpacking-trips', 'https://go4explore.com/trips-category/weekend-trips', 'https://go4explore.com/trips-category/international-trips']
['https://go4explore.com/trips-category/backpacking-trips', 'https://go4explore.com/trips-category/weekend-trips', 'https://go4explore.com/trips-category/international-trips', 'https://go4explore.com/trips-category/trekking']
['https://go4explore.com/trips-category/backpacking-trips', 'https://go4explore.com/trips-category/weekend-trips', 'https://go4explore.com/trips-category/international-trips', 'https://go4explore.com/trips-category/trekking', 'https://go4explore.com/trips-category/honeymoon-packages']
['https://go4explore.com/trips-category/backpacking-trips', 'https://go4explore.com/trips-category/weekend-trips', 'https://go4explore.co

In [44]:
link_length = len(title_link)
print(link_length)

6


In [ ]:
for p in range(link_length):
    try:
        # Navigate to the link
        driver.get(title_link[p])
        view_more_button_xpath = './/button[@class="text-white text-decoration-none border-0 py-2 px-4 fw-medium fs_sm rounded-5 bg_blue"]'

        try:
            while True:
                view_more_button = wait.until(EC.element_to_be_clickable((By.XPATH, view_more_button_xpath)))
                if view_more_button.text.strip() == "View Less":
                    break
                elif view_more_button.text.strip() == "View More":
                    actions = ActionChains(driver)
                    actions.move_to_element(view_more_button).click().perform()
                    # time.sleep(2)  # Wait for 3 seconds to allow content to load

        except Exception as e:
            print(f"An error occurred while clicking 'View More': {e}")

        # Extract the trip names
        data = driver.find_elements(By.XPATH, '//div[contains(@class,"col- col-md-6 col-lg-4 col-xl-4 col-xxl-3 mb-4 px-2 mt-4")]')

        for trip_name_element in data:
            trip_name = trip_name_element.find_element(By.XPATH, './/p[contains(@class, "pt-2 mb-2 text-capitalize font_poppins text-black fw-semibold pkgname_size text_ellipsis ")]').get_attribute("textContent").strip()
            trip_names.append(trip_name)
            print(trip_name)
        
        # Extract the trip price
        for price_element in data:
            try:
                price = price_element.find_element(By.XPATH, './/p[@class="mb-0 d-flex align-items-center text-black fs_12 fw-medium"]').get_attribute("textContent").replace("₹", "").strip()
            except NoSuchElementException:
                price = "N/A"  # Assign default value if price is missing
            prices.append(price)

        # Extract the previous trip price
        for p_price_element in data:
            try:
                p_price = p_price_element.find_element(By.XPATH, './/p[@class="mb-0 text-decoration-line-through clr_gray fs_12 fw-normal"]').get_attribute("textContent").replace("₹", "").strip()
            except NoSuchElementException:
                p_price = "N/A"  # Assign default value if price is missing
            p_prices.append(p_price)

        # Extract duration of the trip
        for duration_element in data:
            try:
                duration = duration_element.find_element(By.XPATH, './/p[@class="bg_yellow white_space rounded-3 position-absolute text-black top-100 start-50 translate-middle fw-normal fs_xsm font_poppins px-2 py-1"]').get_attribute("textContent").strip()
            except NoSuchElementException:
                duration = "N/A"
            durations.append(duration)

        # Extract discount if available
        for discount_element in data:
            try:
                discount = discount_element.find_element(By.XPATH, './/p[@class="bg_red text-white rounded-end-3 position-absolute text-black top-0 mt-4 start-0 fw-medium fs_xsm font_poppins text-center px-3 py-1 d-flex gap-2 align-items-center"]').get_attribute("textContent").replace("Discount:", "").replace("Off", "").replace("₹", "").strip()
                
            except NoSuchElementException:
                discount= "0"  # Default value if discount is missing

            discounts.append(discount)

    except TimeoutException:
        print(f"Timeout: Could not load {title_link[p]}")
        driver.back()  # Go back if page fails to load


Leh Ladakh Backpacking Bike Trip
Spiti Valley Circuit Trip
SM Himachal Backpacking
SM Manali Sissu Kasol
Leh Ladakh With Uming La & Hanle
NY Kashmir Backpacking
NY Meghalaya Backpacking
NY Kasol Kheerganga Manali
NY Uttarakhand Backpacking
Uttarakhand Backpacking- Rishikesh Auli Chopta
Manali Sissu Kasol
Kashmir Backpacking
Himachal Backpacking Summer Special
Leh Srinagar Backpacking Bike Trip
Manali Leh Srinagar Backpacking Bike Trip
Fantastic Goa
Manali Kasol Jibhi
Spiti Valley Circuit Trip with Manali
Ny Shimla Manali
NY Himachal Backpacking
Manali Kasol Jibhi
Spiti Valley Circuit Trip with Manali
Ny Shimla Manali
NY Himachal Backpacking
Leh Ladakh Backpacking Bike Trip
Spiti Valley Circuit Trip
SM Himachal Backpacking
SM Manali Sissu Kasol
Leh Ladakh With Uming La & Hanle
NY Kashmir Backpacking
NY Meghalaya Backpacking
NY Kasol Kheerganga Manali
NY Uttarakhand Backpacking
Uttarakhand Backpacking- Rishikesh Auli Chopta
Manali Sissu Kasol
Kashmir Backpacking
Himachal Backpacking Summ

In [47]:
# close the driver

driver.quit()

In [ ]:
#Store the data into a dataframe and save it as a CSV file

dict_df={"Trip Names": trip_names,"Prices ": p_prices, "Discounted_Prices ": prices, "Discount": discounts, "Duration": durations}

df= pd.DataFrame(dict_df)

df.to_csv("Data/data.csv")